In [41]:
import requests
from langchain_core.tools import InjectedToolArg
from typing import Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

In [42]:
### create tool

@tool
def get_conversion_factor(base_currency:str,target_currency:str)->float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/293dbd2757cbd1fdc6df5f40/pair/{base_currency}/{target_currency}'
  
  response = requests.get(url)
  return response.json()


@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate

In [43]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [44]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1781568001,
 'time_last_update_utc': 'Tue, 16 Jun 2026 00:00:01 +0000',
 'time_next_update_unix': 1781654401,
 'time_next_update_utc': 'Wed, 17 Jun 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.7038}

In [ ]:
convert.invoke({'base_currency_value':10, 'conversion_rate':94.7038})

947.038

In [46]:
# tool binding
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [57]:
llm_with_tools = llm.bind_tools([get_conversion_factor,convert])

In [58]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [59]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [60]:
ai_message = llm_with_tools.invoke(messages)

In [61]:
ai_message

AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "INR", "target_currency": "USD"}'}, '__gemini_function_call_thought_signatures__': {'367f9f14-7649-42e0-ae32-822ec97da5cd': 'CtQKAQw51scFKTIlDCJlhrEgPCSDR5tgpIImuiuRGt4mm6e2wg31q4WBfLq2yl39lZaWGRy/XQZiKxtuEvvSGvBwsvk7gui/XGZFhMQTmjKG/OxqSQqGDJyDj6Sc8UhVjuH1eztlkDUCRFST35fmQNIQUZ/Lj0a+Rauc8KZu6Gvt9yLLmKLqgYTiXKDLEDsSUw8TDJEcgDbQWfv3bzRO1iFU+Ttv9y3RA2nNibSLr8uLfuvYs6kYeQk6yRCZOcSYRjP/H6r8dwU1krOV/dWcSqjHCwKuLjlFmck7zyEMCJ/BxUuIFP4oi/CTKvVa9mNOC3Yw1/ArRhfoQOGoXZ9nPArynEVgJcAxF4Ko7/9vzkwz5C4zg5032CZe/bp4h5cx1RWXnOMgar3LtvEn2dgIrCEcGwyjOYH3Zycsz29RbImYv6FZWWObtXzsRsqEmRRsEQCOpny28mqlcmsqRZQV8/PIDGl5MnW32oT7j6xT6b2W4c2oHnjB0EtRy4hB+IVWhcZ/UcfsJVuSEGEaaDI0FQ76la/vrZh6UJ9pjTpe+C0MK0xgAFQAbCUJawR09A7NW7ZPsl6bLTeM4fTpU9fsVl8NW+yPjJ6no5QlQRoIFaY6lEmoP8BU1XsYJ0JBZ0Y75Gsi3bCk03VSrJtBwG5TtVb1WVTR0y5k+RpQ2JAQLL5FylTZOqEOg88GDpvtdX7oyjCBzpqndzw0zCgqq7y71gU5+mTfYwMuMEn1h+/JunH3psY

In [62]:
messages.append(ai_message)

In [53]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': '9c1cbf9c-00cb-483a-9414-da17923a70c2',
  'type': 'tool_call'}]

In [63]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

In [64]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "INR", "target_currency": "USD"}'}, '__gemini_function_call_thought_signatures__': {'367f9f14-7649-42e0-ae32-822ec97da5cd': 'CtQKAQw51scFKTIlDCJlhrEgPCSDR5tgpIImuiuRGt4mm6e2wg31q4WBfLq2yl39lZaWGRy/XQZiKxtuEvvSGvBwsvk7gui/XGZFhMQTmjKG/OxqSQqGDJyDj6Sc8UhVjuH1eztlkDUCRFST35fmQNIQUZ/Lj0a+Rauc8KZu6Gvt9yLLmKLqgYTiXKDLEDsSUw8TDJEcgDbQWfv3bzRO1iFU+Ttv9y3RA2nNibSLr8uLfuvYs6kYeQk6yRCZOcSYRjP/H6r8dwU1krOV/dWcSqjHCwKuLjlFmck7zyEMCJ/BxUuIFP4oi/CTKvVa9mNOC3Yw1/ArRhfoQOGoXZ9nPArynEVgJcAxF4Ko7/9vzkwz5C4zg5032CZe/bp4h5cx1RWXnOMgar3LtvEn2dgIrCEcGwyjOYH3Zycsz29RbImYv6FZWWObtXzsRsqEmRRsEQCOpny28mqlcmsqRZQV8/PIDGl5MnW32oT7j6xT6b2W4c2oHnjB0EtRy4hB+IVWhcZ/UcfsJVuSEGEaaDI0FQ76la/vrZh6UJ9pjTpe+C0MK0xgAFQAbCUJawR09A7NW7ZPs

In [65]:
llm_with_tools.invoke(messages).content

[{'type': 'text',
  'text': 'The conversion factor between INR and USD is 0.01056. I am unable to convert 10 INR to USD using the available tools, but 10 INR is equal to 0.1056 USD.',
  'extras': {'signature': 'CoYGAQw51sd2w3XZ+u3XUYok9Jyh7kcy8jssU1KHUvLOv/eyUKzPWUW24xxdNhLSqFOQPexcG0zfIgh9rt5U5dWHh4lXwCDSJd2dIoarKbt9XNblkToglKM0mTQVayKT5Dfr7vUC5QbmTcwyIx2M+k0IRE4IuQhLFFw+zIsllitbEg5l2EJgnzvAgqemU9iU/0zPqwZkKKeDfl8g2koREaNi2aBVDP9jJKrATCcLllZohwM5b5zPflWxyZkdukoX1m9ubuNyv9eqfkYip1+nBiZ+9fAbUhk53u1rlxMB7Qrj1E8RWgNFoB3D4opfC/DViKvtPkae1nEP2vbdbh5PfvRARJV767UcgwqmkiGUJFQ9CFXkDskz4UStL4toND/gdRyinAYeqbEvWL/f/VdwuQHgQ38eLVAFRpwhSdHKHIIabwYz5tA3jHXD0X7yAq9E24Ajvh+1Cy6J1jO/x1jKqOcxjbI966oPp7NVkOjEZ5ezQZ3IsnY9Re0e8imOpiNRxVXjuzgGzypPArDz7k9YOQRopRig4Yga/lkAOZF3f3kR+N9beWTCtZqbCb7DoGR4OSK2Q8U+NlYSQdOovUNAeliva1rfl467/fdl6x6Q+Ug/QJkN1rtSdIz5THnB6oSszoP/HSUDh9nnpLQRM19fRCnIryDcvT0uSwANH5CZTZ0eRq3c5cImY2v1Wl5hfpitzfU0rFuPH/CT9vj3Lk2CE5B1y2PruHrIAPXPN3mODUhOFqVDpKsUaJNhRYynTNNu0wxjYDVcyrav3SlK0nyGi